# RelCheck v2 — Viability Test (Pix2Grp + GroundingDINO)

**Goal:** Test scene-graph-grounded correction on 5 images before full rewrite.

**New Architecture:**
1. BLIP-2 generates caption (existing)
2. **Pix2Grp (CVPR 2024)** generates scene graph from image — spatial AND action relations
3. **GroundingDINO** detects objects + bounding boxes (supplements with precise geometry)
4. **Llama-3.3-70B** compares caption claims against scene graph + spatial KB
5. **Llama-3.3-70B** corrects contradicted spans with KB evidence

**Why Pix2Grp:** Unlike bounding-box geometry (which only handles spatial relations),
Pix2Grp outputs (subject, predicate, object) triplets including ACTION relations
(riding, holding, eating, wearing). This is the gap GroundingDINO alone can't fill.

**Decision gate:** Are scene graphs meaningful? Do corrections make sense on ≥3/5 images?

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Install & Setup
# ═══════════════════════════════════════════════════════════════
import os, sys, types, pathlib, shutil, glob

# ── Python deps (only pin: transformers<5 for GroundingDINO BertModel) ──
!pip install -q "transformers>=4.37.0,<5.0.0" accelerate
!pip install -q together supervision spacy
!python -m spacy download en_core_web_sm -q

# ── GroundingDINO ──
if not os.path.exists('/content/GroundingDINO'):
    !git clone -q https://github.com/IDEA-Research/GroundingDINO.git /content/GroundingDINO
!cd /content/GroundingDINO && pip install -e . -q 2>&1 | tail -3
sys.path.insert(0, '/content/GroundingDINO')

os.makedirs('/content/weights', exist_ok=True)
GDINO_WEIGHTS = '/content/weights/groundingdino_swint_ogc.pth'
if not os.path.exists(GDINO_WEIGHTS):
    !wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth -O {GDINO_WEIGHTS}
print('✅ GroundingDINO weights ready')

# ── Pix2Grp / PGSG ──
if not os.path.exists('/content/Pix2Grp'):
    !git clone -q https://github.com/SHTUPLUS/Pix2Grp_CVPR2024.git /content/Pix2Grp

# Patch LAVIS source: make decord optional
data_utils = pathlib.Path('/content/Pix2Grp/lavis/datasets/data_utils.py')
if data_utils.exists():
    code = data_utils.read_text()
    if 'import decord' in code and 'try:' not in code.split('import decord')[0][-30:]:
        code = code.replace(
            'import decord\n',
            'try:\n    import decord\nexcept ImportError:\n    decord = None\n'
        )
        code = code.replace(
            'decord.bridge.set_bridge("torch")',
            'if decord is not None: decord.bridge.set_bridge("torch")'
        )
        data_utils.write_text(code)
        print('  ✅ Patched data_utils.py')

# Patch requirements-dev.txt: strip ALL problematic deps
# CRITICAL: opencv-python==4.8.0.76 replaces Colab's cv2 (compiled for numpy 2.x)
req_path = pathlib.Path('/content/Pix2Grp/requirements-dev.txt')
if req_path.exists():
    lines = req_path.read_text().splitlines()
    skip = {'decord', 'numpy', 'open3d', 'streamlit', 'pre-commit',
            'opencv-python', 'opencv'}  # block ALL opencv pins
    cleaned = [l for l in lines if not any(s in l.lower() for s in skip)]
    req_path.write_text('\n'.join(cleaned) + '\n')
    print(f'  ✅ Patched requirements-dev.txt (stripped {skip})')

# Install LAVIS deps Pix2Grp needs (no opencv — Colab's is fine)
!pip install -q omegaconf einops ftfy iopath webdataset pycocoevalcap \
    pycocotools fairscale==0.4.4 timm==0.4.12 sentencepiece packaging

# Install Pix2Grp — SHOW FULL OUTPUT so we can see if it fails
print('\n--- Pix2Grp pip install -e . ---')
!cd /content/Pix2Grp && pip install -e . --no-build-isolation 2>&1 | tail -20

# Fallback: add to sys.path if editable install failed
sys.path.insert(0, '/content/Pix2Grp')

# Stub decord in sys.modules for runtime imports
import importlib.machinery
decord_stub = types.ModuleType('decord')
decord_stub.__spec__ = importlib.machinery.ModuleSpec('decord', None)
decord_stub.VideoReader = None
decord_stub.cpu = lambda x=0: x
decord_stub.gpu = lambda x=0: x
bridge_stub = types.ModuleType('decord.bridge')
bridge_stub.__spec__ = importlib.machinery.ModuleSpec('decord.bridge', None)
bridge_stub.set_bridge = lambda *a, **k: None
decord_stub.bridge = bridge_stub
sys.modules['decord'] = decord_stub
sys.modules['decord.bridge'] = bridge_stub

# Verify LAVIS
try:
    import lavis
    from lavis.common.registry import registry
    print(f'✅ LAVIS imported from: {lavis.__file__}')
except Exception as e:
    print(f'❌ LAVIS import failed: {e}')
    import traceback; traceback.print_exc()

# ── Mount Drive ──
from google.colab import drive
drive.mount('/content/drive')

os.environ['TOGETHER_API_KEY'] = ''  # ← PASTE YOUR KEY HERE

DRIVE_BASE        = '/content/drive/MyDrive/RelCheck_Data'
DRIVE_EVAL_DIR    = f'{DRIVE_BASE}/eval'
DRIVE_IMAGES_DIR  = f'{DRIVE_BASE}/images'
DRIVE_FIGURES_DIR = f'{DRIVE_BASE}/figures'
DRIVE_MODELS_DIR  = f'{DRIVE_BASE}/hf_cache'

for d in [DRIVE_BASE, DRIVE_EVAL_DIR, DRIVE_IMAGES_DIR, DRIVE_FIGURES_DIR, DRIVE_MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ['HF_HOME']               = DRIVE_MODELS_DIR
os.environ['TRANSFORMERS_CACHE']     = DRIVE_MODELS_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_MODELS_DIR

# ── Clone RelCheck repo ──
REPO_DIR = '/content/RelCheck'
if os.path.exists(f'{REPO_DIR}/.git'):
    !cd {REPO_DIR} && git pull -q
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone -q https://github.com/siddhipatil503/RelCheck.git {REPO_DIR}

# ── Restore eval CSVs from Drive ──
EVAL_DIR = f'{REPO_DIR}/eval'
os.makedirs(EVAL_DIR, exist_ok=True)
restored = 0
for f in glob.glob(f'{DRIVE_EVAL_DIR}/*'):
    fsize = os.path.getsize(f)
    if fsize > 50 * 1024 * 1024:
        continue
    dest = os.path.join(EVAL_DIR, os.path.basename(f))
    if not os.path.exists(dest):
        shutil.copy2(f, dest)
        restored += 1

if restored:
    print(f'✅ Restored {restored} file(s) from Drive')

for needed in ['baseline_no_correction.csv', 'relcheck_results.csv']:
    path = os.path.join(EVAL_DIR, needed)
    print(f'  {"✅" if os.path.exists(path) else "❌"} {needed}')

# ── Verify environment ──
import numpy as np
print(f'\n  numpy: {np.__version__}')
try:
    import cv2
    print(f'  cv2:   {cv2.__version__}')
except Exception as e:
    print(f'  cv2:   BROKEN — {e}')
import transformers
print(f'  transformers: {transformers.__version__}')

print('\n✅ Setup complete')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load Models
# ═══════════════════════════════════════════════════════════════
import torch
import os, sys, yaml

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'PyTorch: {torch.__version__}')

# --- GroundingDINO ---
print('\nLoading GroundingDINO...')
from groundingdino.util.inference import load_model, load_image, predict
GDINO_CONFIG = '/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py'
gdino_model = load_model(GDINO_CONFIG, GDINO_WEIGHTS, device=device)
print('✅ GroundingDINO loaded')

# --- Pix2Grp ---
print('\nLoading Pix2Grp...')
sys.path.insert(0, '/content/Pix2Grp')
pix2grp_model = None
pix2grp_loaded = False
pix2grp_vis_processors = None

try:
    from lavis.common.registry import registry
    from lavis.common.config import Config
    from huggingface_hub import hf_hub_download, list_repo_files

    # --- Step 1: Check what model classes Pix2Grp registered ---
    # Import Pix2Grp's model file so it registers with LAVIS
    from lavis.models.blip2_models.blip2_opt_vrd_xdecoder import Blip2OPTVrd
    print(f'  Blip2OPTVrd class loaded')

    # --- Step 2: Download weights from HuggingFace ---
    HF_REPO = 'rj979797/PGSG-CVPR2024'
    print(f'\n  Downloading weights from {HF_REPO}...')

    hf_files = list_repo_files(HF_REPO)
    print(f'  Available files: {hf_files}')

    ckpt_path = None
    cate_dict_path = None
    for fname in hf_files:
        local = hf_hub_download(HF_REPO, fname)
        if fname.endswith('.pth') or fname.endswith('.pt'):
            ckpt_path = local
            print(f'  Checkpoint: {fname} → {local}')
        if fname.endswith('.json') and 'cate' in fname.lower():
            cate_dict_path = local
            print(f'  Category dict: {fname} → {local}')

    # --- Step 3: Find the right eval config ---
    cfg_candidates = [
        '/content/Pix2Grp/lavis/projects/blip/eval/rel_det_psg_eval.yaml',
        '/content/Pix2Grp/lavis/projects/blip/eval/rel_det_vg_eval.yaml',
    ]
    cfg_path = None
    for c in cfg_candidates:
        if os.path.exists(c):
            cfg_path = c
            break

    if cfg_path:
        with open(cfg_path) as f:
            raw_cfg = yaml.safe_load(f)
        model_cfg = raw_cfg.get('model', {})
        print(f'\n  Config: {os.path.basename(cfg_path)}')
        print(f'  arch={model_cfg.get("arch")}, type={model_cfg.get("model_type")}')

        # Override checkpoint path to our downloaded weights
        if ckpt_path:
            model_cfg['finetuned'] = ckpt_path
        if cate_dict_path:
            model_cfg['cate_dict_url'] = cate_dict_path

        # --- Step 4: Load model ---
        model_cls = registry.get_model_class(model_cfg.get('arch', 'blip2_opt_vrd'))
        if model_cls is None:
            # Try direct class
            model_cls = Blip2OPTVrd
        print(f'  Model class: {model_cls}')

        pix2grp_model = model_cls.from_config(model_cfg)
        pix2grp_model = pix2grp_model.to(device)
        pix2grp_model.eval()
        pix2grp_loaded = True
        print('✅ Pix2Grp model loaded!')

        # Load vis_processors for image preprocessing
        try:
            from lavis.processors import load_processor
            vis_cfg = raw_cfg.get('preprocess', {}).get('vis_processor', {}).get('eval', {})
            if vis_cfg:
                pix2grp_vis_processors = load_processor(vis_cfg.get('name', 'blip_image_eval'))
            else:
                from lavis.processors.blip_processors import BlipImageEvalProcessor
                pix2grp_vis_processors = BlipImageEvalProcessor(image_size=224)
            print('✅ Visual processor ready')
        except Exception as e:
            print(f'  ⚠️ Vis processor load issue: {e}')
            from lavis.processors.blip_processors import BlipImageEvalProcessor
            pix2grp_vis_processors = BlipImageEvalProcessor(image_size=224)
    else:
        print('⚠️ No Pix2Grp config found')

except Exception as e:
    import traceback
    print(f'⚠️ Pix2Grp setup failed: {e}')
    traceback.print_exc()
    print('  Will proceed with BLIP-2 structured prompting fallback in Cell 4')

print(f'\n✅ Model loading complete')
print(f'   GroundingDINO: ✅')
print(f'   Pix2Grp: {"✅" if pix2grp_loaded else "❌ (BLIP-2 fallback)"}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Pick 5 Test Images (diverse relation types)
# ═══════════════════════════════════════════════════════════════
import pandas as pd
import json
import random
import re

df_b1 = pd.read_csv(f'{EVAL_DIR}/baseline_no_correction.csv')
b1_cap_col = 'blip2_caption' if 'blip2_caption' in df_b1.columns else 'original_caption'

df_rc = pd.read_csv(f'{EVAL_DIR}/relcheck_results.csv')
corrected_ids = list(df_rc[df_rc['edit_rate'] > 0]['image_id'].unique())

# Build caption map for ALL corrected images
all_caps = {}
for img_id in corrected_ids:
    rows = df_b1[df_b1['image_id'] == img_id]
    if len(rows) > 0:
        all_caps[img_id] = str(rows.iloc[0][b1_cap_col])

# Categorize by relation type
ACTION_WORDS = {'riding', 'holding', 'carrying', 'eating', 'drinking', 'wearing',
                'playing', 'sitting', 'standing', 'walking', 'running', 'flying',
                'pulling', 'pushing', 'throwing', 'catching', 'climbing', 'jumping',
                'cooking', 'cutting', 'reading', 'watching', 'talking', 'looking',
                'hanging', 'laying', 'lying', 'leaning', 'feeding', 'driving'}
SPATIAL_WORDS = {'on', 'in', 'above', 'below', 'under', 'beside', 'next to',
                 'near', 'behind', 'front', 'between', 'left', 'right', 'top',
                 'inside', 'outside', 'across', 'along', 'through', 'over'}

action_imgs = []
spatial_imgs = []
both_imgs = []

for img_id, cap in all_caps.items():
    words = set(re.findall(r'\w+', cap.lower()))
    has_action = bool(words & ACTION_WORDS)
    has_spatial = bool(words & SPATIAL_WORDS)
    if has_action and has_spatial:
        both_imgs.append(img_id)
    elif has_action:
        action_imgs.append(img_id)
    elif has_spatial:
        spatial_imgs.append(img_id)

print(f'Corrected images: {len(corrected_ids)} total')
print(f'  Action relations:  {len(action_imgs)}')
print(f'  Spatial relations: {len(spatial_imgs)}')
print(f'  Both:              {len(both_imgs)}')

# Show some candidates from each category
print(f'\n--- ACTION candidates (pick 2) ---')
for img_id in action_imgs[:8]:
    print(f'  {img_id}: {all_caps[img_id][:90]}')
print(f'\n--- SPATIAL candidates (pick 2) ---')
for img_id in spatial_imgs[:8]:
    print(f'  {img_id}: {all_caps[img_id][:90]}')
print(f'\n--- BOTH candidates (pick 1) ---')
for img_id in both_imgs[:8]:
    print(f'  {img_id}: {all_caps[img_id][:90]}')

# Auto-select: 2 action + 2 spatial + 1 both (for coverage)
random.seed(42)
test_ids = []
if both_imgs:
    test_ids += random.sample(both_imgs, min(1, len(both_imgs)))
if action_imgs:
    test_ids += random.sample(action_imgs, min(2, len(action_imgs)))
if spatial_imgs:
    test_ids += random.sample(spatial_imgs, min(2, len(spatial_imgs)))
# Fill to 5 if any category was empty
if len(test_ids) < 5:
    remaining = [x for x in corrected_ids if x not in test_ids]
    test_ids += random.sample(remaining, min(5 - len(test_ids), len(remaining)))

test_ids = test_ids[:5]

print(f'\n=== SELECTED TEST IMAGES ({len(test_ids)}) ===')

# Build maps
id_to_caption = {}
id_to_path = {}
for img_id in test_ids:
    id_to_caption[img_id] = all_caps.get(img_id, 'N/A')
    for ext in ['.jpg', '.jpeg', '.png', '.webp']:
        p = os.path.join(DRIVE_IMAGES_DIR, f'{img_id}{ext}')
        if os.path.exists(p):
            id_to_path[img_id] = p
            break

for img_id in test_ids:
    found = '✅' if img_id in id_to_path else '❌'
    cap = id_to_caption.get(img_id, 'N/A')
    # Show relation type
    words = set(re.findall(r'\w+', cap.lower()))
    rtype = []
    if words & ACTION_WORDS: rtype.append('ACTION')
    if words & SPATIAL_WORDS: rtype.append('SPATIAL')
    print(f'  {found} {img_id} [{",".join(rtype) or "OTHER"}]: {cap[:85]}')

print(f'\n⚠️  If these don\'t look diverse enough, manually set test_ids in the next line:')
print(f'    # test_ids = [img_id_1, img_id_2, ...]  # uncomment and set your own')
# test_ids = []  # ← uncomment and fill in to override auto-selection

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Stage A: Pix2Grp Scene Graph Generation
# ═══════════════════════════════════════════════════════════════
#
# Generate scene graphs from images using Pix2Grp.
# Output: (subject, predicate, object) triplets INCLUDING action relations.
#
# If Pix2Grp fails to load, we fall back to a manual approach:
#   GroundingDINO detection + BLIP-2 region captioning
#
from PIL import Image
import numpy as np

all_scene_graphs = {}  # img_id → list of (subject, predicate, object) tuples

# --- Attempt 1: Try Pix2Grp ---
pix2grp_works = False

try:
    # Pix2Grp inference — adapted from their evaluate.py
    # The model generates scene graph triplets as text sequences
    from lavis.common.config import Config
    from lavis.common.registry import registry
    import argparse
    
    # Find the right config
    cfg_path = None
    for candidate in [
        '/content/Pix2Grp/lavis/projects/blip/eval/rel_det_psg_eval.yaml',
        '/content/Pix2Grp/lavis/projects/blip/eval/rel_det_vg_eval.yaml',
    ]:
        if os.path.exists(candidate):
            cfg_path = candidate
            break
    
    if cfg_path:
        print(f'Loading Pix2Grp from config: {os.path.basename(cfg_path)}')
        # Parse config to get model details
        import yaml
        with open(cfg_path) as f:
            raw_cfg = yaml.safe_load(f)
        
        model_cfg = raw_cfg.get('model', {})
        print(f'  Model arch: {model_cfg.get("arch", "?")}') 
        print(f'  Model type: {model_cfg.get("model_type", "?")}')
        print(f'  Checkpoint: {model_cfg.get("finetuned", "?")}')
        
        # Try to load
        arch = model_cfg.get('arch', 'blip2_rel_det')
        model_type = model_cfg.get('model_type', 'pretrain')
        
        # Check if HuggingFace weights are available
        HF_WEIGHTS = 'rj979797/PGSG-CVPR2024'
        print(f'\n  Checking HuggingFace weights: {HF_WEIGHTS}')
        
        from huggingface_hub import hf_hub_download, list_repo_files
        try:
            files = list_repo_files(HF_WEIGHTS)
            print(f'  HF files: {files}')
            # Download the checkpoint
            for f_name in files:
                if f_name.endswith('.pth') or f_name.endswith('.pt'):
                    ckpt_path = hf_hub_download(HF_WEIGHTS, f_name, cache_dir='/content/weights/pix2grp')
                    print(f'  Downloaded: {f_name} → {ckpt_path}')
        except Exception as e:
            print(f'  HF download issue: {e}')
        
        # Try to load model via LAVIS registry
        try:
            model_cls = registry.get_model_class(arch)
            if model_cls:
                print(f'  Found model class: {model_cls}')
                # Load with config
                pix2grp_model = model_cls.from_config(model_cfg)
                pix2grp_model = pix2grp_model.to(device)
                pix2grp_model.eval()
                pix2grp_works = True
                print('✅ Pix2Grp model loaded!')
            else:
                print(f'  Model class "{arch}" not in LAVIS registry')
        except Exception as e:
            print(f'  Model load failed: {e}')
    else:
        print('⚠️ No Pix2Grp config found')

except Exception as e:
    print(f'⚠️ Pix2Grp setup failed: {e}')

# --- Attempt 2: If Pix2Grp failed, use BLIP-2 structured prompting ---
# Ask BLIP-2 to describe relationships in the image
# This is weaker than a dedicated SGG model but better than nothing

if not pix2grp_works:
    print('\n⚠️ Pix2Grp not available. Using BLIP-2 structured prompting as fallback.')
    print('   (This is weaker for action relations — note in results)')
    
    from transformers import Blip2Processor, Blip2ForConditionalGeneration
    
    if 'blip2_model' not in dir():
        print('Loading BLIP-2...')
        blip2_processor = Blip2Processor.from_pretrained('Salesforce/blip2-flan-t5-xl')
        blip2_model = Blip2ForConditionalGeneration.from_pretrained(
            'Salesforce/blip2-flan-t5-xl',
            torch_dtype=torch.float16,
        ).to(device)
        blip2_model.eval()
        print('✅ BLIP-2 loaded')

# --- Generate scene graphs for all 5 images ---
for img_id in test_ids:
    if img_id not in id_to_path:
        continue
    
    img_path = id_to_path[img_id]
    caption = id_to_caption[img_id]
    image = Image.open(img_path).convert('RGB')
    
    print(f'\n{"="*60}')
    print(f'Image: {img_id}')
    print(f'Caption: {caption}')
    
    if pix2grp_works:
        # Use Pix2Grp model to generate scene graph
        # (Exact inference code depends on model API — adapt from their evaluate.py)
        print('  [Using Pix2Grp]')
        # TODO: Run Pix2Grp inference — parse output into (s, p, o) tuples
        scene_graph = []  # placeholder
    else:
        # Fallback: BLIP-2 structured prompting
        # Ask multiple focused questions to build a scene graph
        print('  [Using BLIP-2 structured prompting]')
        
        sg_prompts = [
            'Question: List all objects in this image and their spatial relationships. Answer:',
            'Question: What actions are being performed in this image? Who is doing what to whom? Answer:',
            'Question: Describe the spatial arrangement of objects: what is on top of what, what is next to what, what is inside what? Answer:',
        ]
        
        descriptions = []
        for prompt in sg_prompts:
            inputs = blip2_processor(images=image, text=prompt, return_tensors='pt').to(device, torch.float16)
            with torch.no_grad():
                out = blip2_model.generate(**inputs, max_new_tokens=100)
            desc = blip2_processor.decode(out[0], skip_special_tokens=True).strip()
            descriptions.append(desc)
            print(f'  Q: {prompt[:60]}...')
            print(f'  A: {desc}')
        
        # Store raw descriptions — Llama will parse these in the comparison step
        scene_graph = descriptions  # list of description strings
    
    all_scene_graphs[img_id] = scene_graph

print(f'\n{"="*60}')
print(f'Scene graphs generated for {len(all_scene_graphs)} images')
print(f'Method: {"Pix2Grp" if pix2grp_works else "BLIP-2 structured prompting (fallback)"}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Stage B: GroundingDINO Detection + Spatial Geometry
# ═══════════════════════════════════════════════════════════════
#
# GroundingDINO gives us precise object locations.
# Geometric rules give us deterministic spatial relations.
# This supplements Pix2Grp (or BLIP-2 fallback) with hard spatial facts.
#
import re
import spacy
nlp = spacy.load('en_core_web_sm')

def extract_noun_phrases(caption):
    doc = nlp(caption)
    phrases = []
    for chunk in doc.noun_chunks:
        text = chunk.text.strip()
        text = re.sub(r'^(a|an|the|some|this|that|these|those)\s+', '', text, flags=re.I)
        if text and len(text) > 1:
            phrases.append(text)
    return list(set(phrases))

def detect_objects(image_path, noun_phrases, box_threshold=0.30, text_threshold=0.25):
    image_source, image_tensor = load_image(image_path)
    h, w = image_source.shape[:2]
    text_prompt = ' . '.join(noun_phrases) + ' .'
    boxes, logits, phrases = predict(
        model=gdino_model, image=image_tensor,
        caption=text_prompt, box_threshold=box_threshold, text_threshold=text_threshold,
    )
    detections = []
    for i in range(len(boxes)):
        cx, cy, bw, bh = boxes[i].tolist()
        x1 = int((cx - bw/2) * w)
        y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w)
        y2 = int((cy + bh/2) * h)
        detections.append({'phrase': phrases[i], 'box': [x1, y1, x2, y2],
                           'confidence': round(logits[i].item(), 3)})
    return detections, (w, h)

def compute_spatial_relations(detections, img_size):
    w, h = img_size
    relations = []
    for i, a in enumerate(detections):
        for j, b in enumerate(detections):
            if i == j or a['phrase'].lower() == b['phrase'].lower():
                continue
            ax1, ay1, ax2, ay2 = a['box']
            bx1, by1, bx2, by2 = b['box']
            a_cx, a_cy = (ax1+ax2)/2, (ay1+ay2)/2
            b_cx, b_cy = (bx1+bx2)/2, (by1+by2)/2
            a_area = (ax2-ax1)*(ay2-ay1)
            b_area = (bx2-bx1)*(by2-by1)
            ox1, oy1 = max(ax1,bx1), max(ay1,by1)
            ox2, oy2 = min(ax2,bx2), min(ay2,by2)
            overlap = max(0,ox2-ox1)*max(0,oy2-oy1)
            a_in_b = overlap/a_area if a_area > 0 else 0
            horiz_overlap = max(0, min(ax2,bx2)-max(ax1,bx1))
            a_w = ax2-ax1
            h_ratio = horiz_overlap/a_w if a_w > 0 else 0
            dist = ((a_cx-b_cx)**2+(a_cy-b_cy)**2)**0.5
            diag = (w**2+h**2)**0.5
            iou = overlap/(a_area+b_area-overlap) if (a_area+b_area-overlap) > 0 else 0
            b_h = by2-by1
            s, o = a['phrase'], b['phrase']
            # ON
            if h_ratio > 0.3 and abs(ay2-by1) < 0.3*b_h and a_cy < b_cy:
                relations.append((s, 'on', o))
            # IN
            if a_in_b > 0.6 and a_area < b_area*0.8:
                relations.append((s, 'in', o))
            # ABOVE
            if a_cy < b_cy - 0.1*h and h_ratio > 0.2:
                relations.append((s, 'above', o))
            # LEFT OF
            if a_cx < b_cx - 0.15*w and abs(a_cy-b_cy) < 0.25*h:
                relations.append((s, 'to the left of', o))
            # NEAR
            if dist < 0.25*diag and iou < 0.3:
                relations.append((s, 'near', o))
            # NEXT TO
            if abs(a_cy-b_cy) < 0.1*h and dist < 0.2*diag and iou < 0.15:
                relations.append((s, 'next to', o))
    return list(set(relations))

# --- Run on all 5 images ---
all_spatial_kbs = {}  # img_id → {objects, spatial_relations, missing}

for img_id in test_ids:
    if img_id not in id_to_path:
        continue
    caption = id_to_caption[img_id]
    nps = extract_noun_phrases(caption)
    dets, img_size = detect_objects(id_to_path[img_id], nps)
    spatial_rels = compute_spatial_relations(dets, img_size)
    detected_phrases = {d['phrase'].lower() for d in dets}
    missing = [n for n in nps if not any(n.lower() in dp or dp in n.lower() for dp in detected_phrases)]
    
    all_spatial_kbs[img_id] = {
        'objects': [(d['phrase'], d['confidence']) for d in dets],
        'spatial_relations': spatial_rels,
        'missing_objects': missing,
    }
    
    print(f'\n{"="*60}')
    print(f'Image: {img_id}')
    print(f'Caption: {caption}')
    print(f'Objects ({len(dets)}): {", ".join(d["phrase"] for d in dets)}')
    print(f'Spatial relations ({len(spatial_rels)}):')
    for s, r, o in spatial_rels:
        print(f'  ({s}, {r}, {o})')
    if missing:
        print(f'⚠️ Missing: {missing}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Stage C: Merge KB + Compare Caption (Llama-3.3-70B)
# ═══════════════════════════════════════════════════════════════
#
# Merge scene graph (Pix2Grp or BLIP-2 descriptions) with spatial geometry
# into a single KB, then ask Llama to compare caption against it.
#
import together
import time
import json

client = together.Together(api_key=os.environ.get('TOGETHER_API_KEY'))
LLM_MODEL = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'

COMPARE_SYSTEM = """You are a precise image caption fact-checker. You will receive:
1. An image caption generated by an AI model
2. A knowledge base with TWO sources of evidence:
   a) Scene descriptions — what a vision model sees in the image (objects, actions, relationships)
   b) Spatial facts — deterministic spatial relations computed from object bounding boxes

Your job: identify relational claims in the caption that CONTRADICT the evidence.

Rules:
- A contradiction requires POSITIVE evidence against the claim (e.g., KB says "cat on floor" but caption says "cat on table")
- If the KB simply doesn't mention something, that is NOT a contradiction
- Spatial facts (from geometry) are MORE RELIABLE than scene descriptions (from a vision model)
- When spatial facts and scene descriptions disagree, trust spatial facts
- Flag action relations (holding, riding, eating) as contradictions ONLY if scene descriptions clearly contradict them
- Be conservative — only flag clear contradictions, not ambiguities

Output format (strict JSON):
{
  "contradictions": [
    {"caption_claim": "...", "evidence": "...", "source": "spatial|scene_description", "confidence": "high|medium"}
  ],
  "supported": [
    {"caption_claim": "...", "evidence": "..."}
  ],
  "unverifiable": [
    {"caption_claim": "...", "reason": "..."}
  ]
}"""

COMPARE_USER = """Caption: "{caption}"

=== KNOWLEDGE BASE ===

Scene descriptions (from vision model):
{scene_descriptions}

Spatial facts (from bounding box geometry — high reliability):
- Detected objects: {objects}
- Spatial relations: {spatial_relations}
- Objects in caption but NOT detected: {missing}

=== TASK ===
Compare the caption against BOTH sources. Identify contradictions, supported claims, and unverifiable claims.
Output ONLY valid JSON."""

def compare_caption(caption, scene_graph, spatial_kb, max_retries=3):
    # Format scene descriptions
    if isinstance(scene_graph, list) and scene_graph and isinstance(scene_graph[0], str):
        scene_str = '\n'.join(f'  - {desc}' for desc in scene_graph)
    elif isinstance(scene_graph, list) and scene_graph and isinstance(scene_graph[0], tuple):
        scene_str = '\n'.join(f'  - ({s}, {p}, {o})' for s, p, o in scene_graph)
    else:
        scene_str = '  (none available)'
    
    objects_str = ', '.join(f'{obj} (conf={conf})' for obj, conf in spatial_kb['objects'])
    spatial_str = ', '.join(f'({s}, {r}, {o})' for s, r, o in spatial_kb['spatial_relations'])
    missing_str = ', '.join(spatial_kb['missing_objects']) if spatial_kb['missing_objects'] else 'None'
    
    user_msg = COMPARE_USER.format(
        caption=caption,
        scene_descriptions=scene_str,
        objects=objects_str or 'None detected',
        spatial_relations=spatial_str or 'None computed',
        missing=missing_str,
    )
    
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {'role': 'system', 'content': COMPARE_SYSTEM},
                    {'role': 'user', 'content': user_msg},
                ],
                max_tokens=800, temperature=0.1,
            )
            raw = resp.choices[0].message.content.strip()
            if '```json' in raw:
                raw = raw.split('```json')[1].split('```')[0].strip()
            elif '```' in raw:
                raw = raw.split('```')[1].split('```')[0].strip()
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f'  ⚠️ JSON parse failed (attempt {attempt+1})')
            if attempt < max_retries - 1: time.sleep(1)
        except Exception as e:
            print(f'  ⚠️ API error: {e}')
            if attempt < max_retries - 1: time.sleep(2**attempt)
    return {'contradictions': [], 'supported': [], 'unverifiable': [], 'error': True}

# --- Run comparison ---
all_comparisons = {}

for img_id in test_ids:
    if img_id not in all_spatial_kbs or img_id not in all_scene_graphs:
        continue
    caption = id_to_caption[img_id]
    result = compare_caption(caption, all_scene_graphs[img_id], all_spatial_kbs[img_id])
    all_comparisons[img_id] = result
    
    n_c = len(result.get('contradictions', []))
    n_s = len(result.get('supported', []))
    n_u = len(result.get('unverifiable', []))
    
    print(f'\n{"="*60}')
    print(f'Image: {img_id}')
    print(f'Caption: {caption}')
    print(f'Results: {n_c} contradictions, {n_s} supported, {n_u} unverifiable')
    
    for c in result.get('contradictions', []):
        print(f'  ❌ "{c.get("caption_claim","?")}" — Evidence: {c.get("evidence","?")} [{c.get("source","?")}]')
    for s in result.get('supported', []):
        print(f'  ✅ "{s.get("caption_claim","?")}"')

total_c = sum(len(r.get('contradictions',[])) for r in all_comparisons.values())
print(f'\n{"="*60}')
print(f'Total contradictions: {total_c}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Stage D: Correction (3 methods compared)
# ═══════════════════════════════════════════════════════════════

CORRECT_STRUCTURED_SYS = """You are a precise caption editor. Fix ONLY the contradicted claims listed below.
Keep everything else exactly the same. If no contradictions, return caption unchanged.
Output ONLY the corrected caption."""

CORRECT_KB_DUMP_SYS = """You are a precise caption editor. Fix any relational claims that contradict the knowledge base.
Make minimal changes. If nothing contradicts, return unchanged.
Output ONLY the corrected caption."""

B3_SYS = """You are a precise image caption editor. Fix any incorrect relationships or spatial descriptions.
Make minimal changes. If it seems fine, return unchanged. Output ONLY the corrected caption."""

def correct(caption, method, kb=None, comparison=None, scene_graph=None):
    if method == 'b3':
        msgs = [{'role':'system','content':B3_SYS},
                {'role':'user','content':f'Caption: "{caption}"\n\nFix any incorrect relationships:'}]
    elif method == 'structured':
        contras = comparison.get('contradictions', [])
        if not contras: return caption
        contra_str = '\n'.join(f'- "{c["caption_claim"]}" contradicts: {c["evidence"]}' for c in contras)
        msgs = [{'role':'system','content':CORRECT_STRUCTURED_SYS},
                {'role':'user','content':f'Caption: "{caption}"\n\nContradictions:\n{contra_str}\n\nFix only these:'}]
    elif method == 'kb_dump':
        obj_str = ', '.join(o for o,c in kb['objects'])
        rel_str = ', '.join(f'({s},{r},{o})' for s,r,o in kb['spatial_relations']) or 'None'
        scene_str = '\n'.join(scene_graph) if isinstance(scene_graph, list) and scene_graph and isinstance(scene_graph[0], str) else 'N/A'
        msgs = [{'role':'system','content':CORRECT_KB_DUMP_SYS},
                {'role':'user','content':f'Caption: "{caption}"\n\nKB:\nObjects: {obj_str}\nSpatial: {rel_str}\nScene: {scene_str}\n\nFix contradictions:'}]
    else:
        return caption
    
    try:
        resp = client.chat.completions.create(model=LLM_MODEL, messages=msgs, max_tokens=300, temperature=0.2)
        out = resp.choices[0].message.content.strip().strip('"').strip("'").strip()
        if len(out) < len(caption)*0.3 or len(out) > len(caption)*2: return caption
        return out
    except:
        return caption

def edit_rate(a, b):
    if a == b: return 0.0
    m, n = len(a), len(b)
    dp = list(range(n+1))
    for i in range(1, m+1):
        prev = dp[0]; dp[0] = i
        for j in range(1, n+1):
            tmp = dp[j]
            dp[j] = prev if a[i-1]==b[j-1] else 1+min(prev, dp[j], dp[j-1])
            prev = tmp
    return dp[n]/max(m,1)

# --- Run all 3 methods ---
results = []
for img_id in test_ids:
    if img_id not in all_comparisons: continue
    cap = id_to_caption[img_id]
    kb = all_spatial_kbs[img_id]
    comp = all_comparisons[img_id]
    sg = all_scene_graphs.get(img_id, [])
    
    b3 = correct(cap, 'b3')
    kd = correct(cap, 'kb_dump', kb=kb, scene_graph=sg)
    st = correct(cap, 'structured', comparison=comp)
    
    print(f'\n{"="*60}')
    print(f'Image: {img_id}')
    print(f'Original:    {cap}')
    print(f'B3:          {b3}  (edit={edit_rate(cap,b3):.3f})')
    print(f'KB dump:     {kd}  (edit={edit_rate(cap,kd):.3f})')
    print(f'Structured:  {st}  (edit={edit_rate(cap,st):.3f})')
    
    results.append({'image_id': img_id, 'original': cap, 'b3': b3, 'kb_dump': kd, 'structured': st,
                    'n_contradictions': len(comp.get('contradictions',[])),
                    'n_supported': len(comp.get('supported',[])),
                    'n_unverifiable': len(comp.get('unverifiable',[]))})

print(f'\n{"="*60}')
print('SUMMARY')
for r in results:
    print(f'{r["image_id"]}: {r["n_contradictions"]} contras | '
          f'B3={"✏️" if r["b3"]!=r["original"] else "—"} '
          f'KB={"✏️" if r["kb_dump"]!=r["original"] else "—"} '
          f'Struct={"✏️" if r["structured"]!=r["original"] else "—"}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Viability Verdict + Save
# ═══════════════════════════════════════════════════════════════

print('\n' + '='*75)
print('  VIABILITY ASSESSMENT')
print('='*75)

print(f'\n  Scene graph method: {"Pix2Grp" if pix2grp_works else "BLIP-2 structured prompting"}')
print(f'  Spatial KB method:  GroundingDINO + bounding box geometry')

print('\n📋 Key Questions:')
print('  1. Did scene descriptions capture ACTION relations? (riding, holding, eating)')
print('  2. Did spatial geometry add useful facts beyond scene descriptions?')
print('  3. Were contradictions REAL (not hallucinated by Llama)?')
print('  4. Are structured corrections better than blind B3 corrections?')
print('  5. Are corrected captions grammatically correct and faithful?')

n_contras = sum(1 for r in results if r['n_contradictions'] > 0)
n_changed = sum(1 for r in results if r['structured'] != r['original'])
n = len(results)

print(f'\n  Images with contradictions: {n_contras}/{n}')
print(f'  Images with corrections:    {n_changed}/{n}')

if pix2grp_works:
    print('\n  ✅ Pix2Grp loaded — scene graph quality determines viability')
else:
    print('\n  ⚠️ Using BLIP-2 fallback — action relations may be weak')
    print('     If results look good even with fallback, Pix2Grp will be better')
    print('     If results look bad, need to fix Pix2Grp setup before proceeding')

if n_contras >= 3:
    print('\n  ✅ VIABLE — proceed with full pipeline')
elif n_contras >= 1:
    print('\n  ⚠️ MARGINAL — check correction quality carefully')
else:
    print('\n  ❌ NOT VIABLE — investigate KB quality')

# Save
output = {
    'test_date': '2026-03-21',
    'scene_graph_method': 'pix2grp' if pix2grp_works else 'blip2_structured',
    'n_images': n,
    'scene_graphs': {k: v if isinstance(v, list) and v and isinstance(v[0], str) else [str(x) for x in v] for k, v in all_scene_graphs.items()},
    'spatial_kbs': {k: {'objects': v['objects'], 'spatial_relations': [list(r) for r in v['spatial_relations']], 'missing': v['missing_objects']} for k, v in all_spatial_kbs.items()},
    'comparisons': all_comparisons,
    'corrections': results,
}
out_path = f'{EVAL_DIR}/viability_test_v2.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f'\n💾 Saved to {out_path}')
if os.path.exists(DRIVE_EVAL_DIR):
    import shutil
    shutil.copy2(out_path, DRIVE_EVAL_DIR)
    print(f'💾 Synced to Drive')